## LoRa Coverage Prediction and A* Path Optimization with PyTorch

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import networkx as nx
from scipy.spatial import distance
import folium
from folium.plugins import HeatMap
from heapq import heappush, heappop
import warnings
import ee
import os
from dotenv import load_dotenv
from geopy.distance import geodesic
import time
warnings.filterwarnings('ignore')

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"GPU Available: {torch.cuda.is_available()}")

### Initializing Google Earth Engine and variables

In [ ]:
# Load environment variables
load_dotenv()

# Initialize Earth Engine
try:
    ee.Initialize(project=os.getenv('GEE_PROJECT_ID'))
    print("Google Earth Engine initialized successfully")
except Exception as e:
    print(f"Google Earth Engine initialization failed: {str(e)}")
    print("Please make sure you have authenticated with GEE and have an active project")
    exit()

# Land cover labels
LAND_COVER_LABELS = {
    10: 'Tree',
    20: 'Shrubland',
    30: 'Grassland',
    40: 'Cropland',
    50: 'Built-up',
    60: 'Bare / sparse vegetation',
    70: 'Snow and ice',
    80: 'Permanent water bodies',
    90: 'Herbaceous wetland',
    95: 'Mangroves',
    100: 'Moss and lichen'
}

# Terrain penalty factors
PENALTY_MAP = {
    10: 0.9,
    20: 0.85,
    30: 0.8,
    40: 0.75,
    50: 0.6,
    60: 0.7,
    70: 0.95,
    80: 0.95,
    90: 0.85,
    95: 0.9,
    100: 0.95
}

# SNR threshold for LoRa
SNR_THRESHOLD = {
    7: -7.5,  # SF7
    8: -10,   # SF8
    9: -12.5, # SF9
    10: -15,  # SF10
    11: -17.5, # SF11
    12: -20   # SF12
}

# Map land cover to k value
LAND_COVER_TO_K = {
    10: 0.25,  # Tree cover → dense foliage
    20: 0.28,  # Shrubland
    30: 0.32,  # Grassland
    40: 0.33,  # Cropland
    50: 0.20,  # Built-up → severe multipath
    60: 0.40,  # Bare/sparse → near-LOS
    70: 0.35,  # Snow/ice → reflective but open
    80: 0.45,  # Water → excellent for LoRa (over ocean)
    90: 0.22,  # Wetland
    95: 0.20,  # Mangroves → worst case
    100: 0.30  # Moss/lichen (tundra)
}

### Data Loading and Preprocessing

In [ ]:
def load_and_preprocess_data():
    """Load and preprocess both datasets"""
    print("Loading and preprocessing data...")
    
    # Load the datasets
    data1 = pd.read_csv('../data/processed_data_1.csv')
    data2 = pd.read_csv('../data/processed_data_2.csv')
    
    # Ensure consistent column names
    if 'land_cover' not in data1.columns and 'land_cover_code' in data1.columns:
        data1 = data1.rename(columns={'land_cover_code': 'land_cover'})
    
    if 'land_cover' not in data2.columns and 'land_cover_code' in data2.columns:
        data2 = data2.rename(columns={'land_cover_code': 'land_cover'})
    
    #  Print shape and column names
    print(f"\nDataset 1 shape: {data1.shape}")
    print(f"Dataset 2 shape: {data2.shape}")
    print(f"\nDataset 1 columns: {data1.columns.tolist()}")
    print(f"Dataset 2 columns: {data2.columns.tolist()}")
    
    # Select only the columns we need for training
    columns_to_keep = [
        'elevation', 'land_cover', 'terrain_penalty', 
        'distance_to_start', 'distance_to_destination',
        'PDR', 'RSSI', 'SNR', 'observed_path_loss'
    ]
    
    # Filter columns that exist in each dataset
    data1_cols = [col for col in columns_to_keep if col in data1.columns]
    data2_cols = [col for col in columns_to_keep if col in data2.columns]
    
    data1_filtered = data1[data1_cols].copy()
    data2_filtered = data2[data2_cols].copy()
    
    # Combine the datasets
    combined_data = pd.concat([data1_filtered, data2_filtered], ignore_index=True)
    combined_data = combined_data.dropna()
    
    print(f"\nCombined dataset shape: {combined_data.shape}")
    print(f"Columns in combined dataset: {combined_data.columns.tolist()}")
    return combined_data

# Load data
combined_data = load_and_preprocess_data()

# Select features and targets
features = ['elevation', 'land_cover', 'terrain_penalty', 
            'distance_to_start', 'distance_to_destination']
targets = ['PDR', 'RSSI', 'SNR', 'observed_path_loss']

X = combined_data[features]
y = combined_data[targets]

# Split and scale data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

print(f"\nModel features: {features}")
print(f"Model targets: {targets}")

### Pytorch Neural Network

In [ ]:
class LoRaDataset(Dataset):
    """Custom dataset for LoRa data"""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LoRaNeuralNetwork(nn.Module):
    """PyTorch Neural Network for LoRa prediction"""
    def __init__(self, input_size, output_size):
        super(LoRaNeuralNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, output_size)
        )
    
    def forward(self, x):
        return self.network(x)

def train_pytorch_model(X_train, y_train, X_test, y_test, input_size, output_size, 
                       epochs=100, batch_size=32, learning_rate=0.001):
    """Train PyTorch neural network"""
    
    # Create datasets and dataloaders
    train_dataset = LoRaDataset(X_train, y_train)
    test_dataset = LoRaDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # Initialize model
    model = LoRaNeuralNetwork(input_size, output_size).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Training loop
    train_losses = []
    test_losses = []
    
    print("Training PyTorch Neural Network...")
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                test_loss += loss.item()
        
        avg_test_loss = test_loss / len(test_loader)
        test_losses.append(avg_test_loss)
        
        if (epoch + 1) % 20 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, Test Loss: {avg_test_loss:.4f}')
    
    # Plot training history
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('PyTorch Neural Network Training History')
    plt.legend()
    plt.grid(True)
    plt.savefig('pytorch_training_history.png')
    plt.show()
    
    return model

### Model Training

In [ ]:
def train_all_models():
    """Train all three models using physically meaningful features"""
    print("\n=== Training Models with Physically Meaningful Features ===")
    
    models = {}
    
    # 1. Random Forest
    print("Training Random Forest...")
    rf_model = RandomForestRegressor(
        n_estimators=100, 
        random_state=42, 
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    models['Random Forest'] = rf_model
    
    # 2. XGBoost
    print("Training XGBoost...")
    xgb_model = xgb.XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        tree_method='hist'
    )
    xgb_model.fit(X_train, y_train)
    models['XGBoost'] = xgb_model
    
    # 3. PyTorch Neural Network
    print("Training PyTorch Neural Network...")
    pytorch_model = train_pytorch_model(
        X_train_scaled, y_train_scaled, 
        X_test_scaled, y_test_scaled,
        input_size=X_train_scaled.shape[1],
        output_size=y_train_scaled.shape[1],
        epochs=100,
        batch_size=32
    )
    models['Neural Network'] = pytorch_model
    
    return models

# Train all models
models = train_all_models()

### Model Evaluation

In [ ]:
def evaluate_all_models():
    """Evaluate all three models"""
    print("\n=== Model Evaluation ===")
    
    evaluation_results = {}
    
    for model_name, model in models.items():
        print(f"\nEvaluating {model_name}...")
        
        if model_name == 'Neural Network':
            # PyTorch evaluation
            model.eval()
            with torch.no_grad():
                X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
                y_pred_scaled = model(X_test_tensor).cpu().numpy()
                y_pred = y_scaler.inverse_transform(y_pred_scaled)
        else:
            # Traditional ML evaluation
            y_pred = model.predict(X_test)
        
        # Calculate metrics for each target
        model_results = {}
        for i, target in enumerate(targets):
            mse = mean_squared_error(y_test[target], y_pred[:, i])
            rmse = np.sqrt(mse)
            mae = mean_absolute_error(y_test[target], y_pred[:, i])
            r2 = r2_score(y_test[target], y_pred[:, i])
            
            model_results[target] = {
                'MSE': mse,
                'RMSE': rmse,
                'MAE': mae,
                'R2': r2
            }
            
            print(f"{target} - MSE: {mse:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")
        
        evaluation_results[model_name] = model_results
    
    # Create evaluation comparison table
    eval_data = []
    for model_name, model_results in evaluation_results.items():
        for target, metrics in model_results.items():
            eval_data.append({
                'Model': model_name,
                'Target': target,
                'MSE': metrics['MSE'],
                'RMSE': metrics['RMSE'],
                'MAE': metrics['MAE'],
                'R2': metrics['R2']
            })
    
    eval_df = pd.DataFrame(eval_data)
    print("\n=== Evaluation Summary ===")
    print(eval_df.to_string(index=False))
    
    # Save evaluation results
    eval_df.to_csv('model_evaluation.csv', index=False)
    
    # Plot evaluation comparison
    plt.figure(figsize=(15, 10))
    metrics = ['MSE', 'RMSE', 'MAE', 'R2']
    for i, metric in enumerate(metrics):
        plt.subplot(2, 2, i+1)
        sns.barplot(x='Target', y=metric, hue='Model', data=eval_df)
        plt.title(f'{metric.upper()} Comparison')
        plt.tight_layout()
    
    plt.savefig('model_evaluation_comparison.png')
    plt.show()
    
    return evaluation_results

# Evaluate all models
evaluation_results = evaluate_all_models()

### Feature Importance Analysis

In [ ]:
def plot_feature_importance():
    """Plot feature importance for tree-based models"""
    plt.figure(figsize=(15, 5))
    
    # Random Forest feature importance
    plt.subplot(1, 2, 1)
    rf_importance = pd.DataFrame({
        'feature': features,
        'importance': models['Random Forest'].feature_importances_
    })
    rf_importance = rf_importance.sort_values('importance', ascending=False)
    sns.barplot(x='importance', y='feature', data=rf_importance)
    plt.title('Random Forest Feature Importance')
    plt.tight_layout()
    
    # XGBoost feature importance
    plt.subplot(1, 2, 2)
    xgb_importance = pd.DataFrame({
        'feature': features,
        'importance': models['XGBoost'].feature_importances_
    })
    xgb_importance = xgb_importance.sort_values('importance', ascending=False)
    sns.barplot(x='importance', y='feature', data=xgb_importance)
    plt.title('XGBoost Feature Importance')
    
    plt.tight_layout()
    plt.savefig('feature_importance.png')
    plt.show()

# Plot feature importance
plot_feature_importance()

### Fetching Real-World Data

In [ ]:
class RealWorldDataFetcher:
    """Class to fetch real-world elevation and land cover data"""
    
    def __init__(self):
        self.elevation_cache = {}
        self.land_cover_cache = {}
    
    def get_elevation(self, lat, lon):
        """Get elevation from Google Earth Engine using SRTM data"""
        try:
            # Create a cache key
            cache_key = f"{lat:.4f}_{lon:.4f}"
            
            # Check cache first
            if cache_key in self.elevation_cache:
                return self.elevation_cache[cache_key]
            
            # Create a point geometry
            point = ee.Geometry.Point([lon, lat])
            
            # Get SRTM elevation data (30m resolution)
            srtm = ee.Image('USGS/SRTMGL1_003')
            
            # Sample elevation at the point
            elevation = srtm.sample(point, 30).first().get('elevation').getInfo()
            
            # Cache the result
            self.elevation_cache[cache_key] = elevation
            return elevation
        except Exception as e:
            print(f"Error getting elevation from GEE: {e}")
            return 0  # Default elevation
    
    def get_land_cover(self, lat, lon):
        """Get land cover from Google Earth Engine using ESA WorldCover"""
        try:
            # Create a cache key
            cache_key = f"{lat:.4f}_{lon:.4f}"
            
            # Check cache first
            if cache_key in self.land_cover_cache:
                return self.land_cover_cache[cache_key]
            
            # Create a point geometry
            point = ee.Geometry.Point([lon, lat])
            
            # Get ESA WorldCover data (10m resolution)
            worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()
            
            # Sample land cover at the point
            land_cover = worldcover.sample(point, 10).first().get('Map').getInfo()
            
            # Cache the result
            self.land_cover_cache[cache_key] = land_cover
            return land_cover
        except Exception as e:
            print(f"Error getting land cover from GEE: {e}")
            return 50  # Default to built-up
    
    def get_terrain_penalty(self, land_cover):
        """Get terrain penalty based on land cover type"""
        return PENALTY_MAP.get(land_cover, 0.5)  # Default penalty
    
    def get_real_world_data(self, lat, lon):
        """Get all real-world data for a point"""
        elevation = self.get_elevation(lat, lon)
        land_cover = self.get_land_cover(lat, lon)
        terrain_penalty = self.get_terrain_penalty(land_cover)
        
        return {
            'elevation': elevation,
            'land_cover': land_cover,
            'terrain_penalty': terrain_penalty
        }

# Initialize the data fetcher
data_fetcher = RealWorldDataFetcher()

### Utility Functions

In [ ]:
def calculate_theoretical_path_loss(distance_m, freq_mhz=868, land_cover_code=None):
    """
    Calculate theoretical path loss using Free-Space Path Loss (FSPL) 
    plus environment-specific excess loss (in dB).
    """
    if distance_m <= 0:
        return np.inf
    
    # Clamp distance to avoid log(0)
    distance_m = max(distance_m, 1.0)
    
    # Free Space Path Loss (FSPL) calculation
    fspl = 20 * np.log10(distance_m) + 20 * np.log10(freq_mhz) - 27.55
    
    # Excess loss (dB) based on land cover
    excess_loss_db = {
        10: 12,  # Tree
        20: 10,  # Shrubland
        30: 6,   # Grassland
        40: 5,   # Cropland
        50: 20,  # Built-up (harbor: metal containers, cranes)
        60: 4,   # Bare/sparse
        70: 8,   # Snow/ice
        80: 5,   # Water
        90: 11,  # Wetland
        95: 15,  # Mangroves
        100: 7   # Moss/lichen
    }.get(land_cover_code, 12)  # Default: urban-like
    
    return fspl + excess_loss_db

def calculate_rssi_from_path_loss(path_loss_db, tx_power_dbm=14):
    """Calculate RSSI from path loss"""
    return tx_power_dbm - path_loss_db

def snr_to_pdr(snr, spreading_factor=7, land_cover_code=None):
    """Convert SNR to PDR for LoRa"""  
    snr_threshold = SNR_THRESHOLD.get(spreading_factor, -7.5)
    margin = snr - snr_threshold
    
    if margin <= 0:
        return 0.0
    
    # Default k if land cover unknown
    k = LAND_COVER_TO_K.get(land_cover_code, 0.3)  # conservative default
    
    pdr = 1 - np.exp(-k * margin)
    return max(0.0, min(1.0, pdr))

def calculate_snr_from_rssi(rssi, noise_floor=-120):
    """Calculate SNR from RSSI"""
    return rssi - noise_floor

### A* Path Optimization

In [ ]:
class AStarPathOptimizer:
    """A* algorithm-based path optimizer for LoRa communication with real-world data"""
    
    def __init__(self, model, scaler, y_scaler, model_name):
        self.model = model
        self.scaler = scaler
        self.y_scaler = y_scaler
        self.model_name = model_name
        
        # Initialize the real-world data fetcher
        self.data_fetcher = RealWorldDataFetcher()
    
    def predict_metrics_at_point(self, lat, lon, start_lat, start_lon, dest_lat, dest_lon,
                                spreading_factor=7, frequency_mhz=868, tx_power_dbm=14):
        """Predict communication metrics at a specific point using real-world data"""
        
        # Get real-world data for this point
        real_data = self.data_fetcher.get_real_world_data(lat, lon)
        elevation = real_data['elevation']
        land_cover = real_data['land_cover']
        terrain_penalty = real_data['terrain_penalty']
        
        # Calculate distances
        dist_to_start = geodesic((lat, lon), (start_lat, start_lon)).meters
        dist_to_destination = geodesic((lat, lon), (dest_lat, dest_lon)).meters
        
        # Create feature array with ONLY physical factors from real-world data
        features = np.array([[
            elevation, land_cover, terrain_penalty, 
            dist_to_start, dist_to_destination
        ]])
        
        if self.model_name == 'Neural Network':
            # PyTorch prediction
            features = self.scaler.transform(features)
            features_tensor = torch.FloatTensor(features).to(device)
            
            self.model.eval()
            with torch.no_grad():
                prediction_scaled = self.model(features_tensor).cpu().numpy()
            
            prediction = self.y_scaler.inverse_transform(prediction_scaled)[0]
        else:
            # Traditional ML model prediction
            prediction = self.model.predict(features)[0]
        
        # Calculate additional metrics based on user inputs
        path_loss = calculate_theoretical_path_loss(dist_to_start, frequency_mhz, land_cover)
        rssi = calculate_rssi_from_path_loss(path_loss, tx_power_dbm)
        snr = calculate_snr_from_rssi(rssi)
        pdr = snr_to_pdr(snr, spreading_factor, land_cover)
        
        return {
            'PDR': prediction[0],  # Use model prediction for PDR
            'RSSI': prediction[1],  # Use model prediction for RSSI
            'SNR': prediction[2],   # Use model prediction for SNR
            'path_loss': prediction[3],  # Use model prediction for path loss
            'elevation': elevation,
            'land_cover': land_cover,
            'terrain_penalty': terrain_penalty,
            'distance_to_start': dist_to_start,
            'distance_to_destination': dist_to_destination,
            'theoretical_path_loss': path_loss,
            'theoretical_rssi': rssi,
            'theoretical_snr': snr,
            'theoretical_pdr': pdr
        }
    
    def heuristic(self, node1, node2):
        """Heuristic function for A* (Euclidean distance)"""
        return distance.euclidean(node1, node2)
    
    def get_neighbors(self, node_idx, grid_shape):
        """Get valid neighboring grid cells"""
        rows, cols = grid_shape
        row, col = node_idx // cols, node_idx % cols
        neighbors = []
        
        # 8-directional movement
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0:
                    continue
                
                nr, nc = row + dr, col + dc
                if 0 <= nr < rows and 0 <= nc < cols:
                    neighbors.append(nr * cols + nc)
        
        return neighbors
    
    def a_star_search(self, start_idx, goal_idx, grid_points, grid_metrics, connectivity_radius=0.003):
        """A* pathfinding algorithm"""
        
        # Priority queue: (f_score, counter, node_idx, path)
        counter = 0
        open_set = [(0, counter, start_idx, [start_idx])]
        closed_set = set()
        
        # g_score: cost from start to node
        g_score = {start_idx: 0}
        
        # f_score: g_score + heuristic
        f_score = {start_idx: self.heuristic(grid_points[start_idx], grid_points[goal_idx])}
        
        while open_set:
            current_f, _, current_idx, path = heappop(open_set)
            
            if current_idx == goal_idx:
                return path
            
            if current_idx in closed_set:
                continue
            
            closed_set.add(current_idx)
            
            # Explore neighbors
            for neighbor_idx in self.get_neighbors(current_idx, 
                                                (int(np.sqrt(len(grid_points))), 
                                                 int(np.sqrt(len(grid_points))))):
                if neighbor_idx in closed_set:
                    continue
                
                # Calculate cost to neighbor
                current_metrics = grid_metrics[current_idx]
                neighbor_metrics = grid_metrics[neighbor_idx]
                
                # Cost based on communication quality (lower is better)
                avg_pdr = (current_metrics['PDR'] + neighbor_metrics['PDR']) / 2
                avg_rssi = (current_metrics['RSSI'] + neighbor_metrics['RSSI']) / 2
                
                # Communication cost (inverse of quality)
                comm_cost = (1 - avg_pdr) * 0.7 + (abs(avg_rssi) / 150) * 0.3
                
                # Distance cost
                dist_cost = distance.euclidean(grid_points[current_idx], 
                                             grid_points[neighbor_idx])
                
                tentative_g_score = g_score[current_idx] + comm_cost + dist_cost
                
                if neighbor_idx not in g_score or tentative_g_score < g_score[neighbor_idx]:
                    g_score[neighbor_idx] = tentative_g_score
                    f_score[neighbor_idx] = tentative_g_score + self.heuristic(
                        grid_points[neighbor_idx], grid_points[goal_idx])
                    
                    counter += 1
                    heappush(open_set, (f_score[neighbor_idx], counter, neighbor_idx, 
                                       path + [neighbor_idx]))
        
        # No path found
        return None
    
    def create_grid(self, min_lat, max_lat, min_lon, max_lon, resolution=0.002):
        """Create a grid of points for optimization"""
        lat_points = np.arange(min_lat, max_lat, resolution)
        lon_points = np.arange(min_lon, max_lon, resolution)
        grid = []
        
        for lat in lat_points:
            for lon in lon_points:
                grid.append((lat, lon))
        
        return np.array(grid)
    
    def find_optimal_path_astar(self, start_lat, start_lon, dest_lat, dest_lon,
                                spreading_factor=7, frequency_mhz=868, tx_power_dbm=14,
                                grid_resolution=0.002):
        """Find optimal path using A* algorithm with real-world data"""
        
        print(f"\nFinding optimal path using {self.model_name} and A* algorithm...")
        print("Using real-world elevation and land cover data...")
        print(f"Parameters: SF={spreading_factor}, Freq={frequency_mhz}MHz, TxPower={tx_power_dbm}dBm")
        
        # Create grid covering the area
        min_lat = min(start_lat, dest_lat) - 0.01
        max_lat = max(start_lat, dest_lat) + 0.01
        min_lon = min(start_lon, dest_lon) - 0.01
        max_lon = max(start_lon, dest_lon) + 0.01
        
        grid_points = self.create_grid(min_lat, max_lat, min_lon, max_lon, grid_resolution)
        print(f"Created grid with {len(grid_points)} points")
        
        # Predict metrics for all grid points using real-world data
        print("Fetching real-world data and predicting communication metrics...")
        grid_metrics = []
        
        for i, point in enumerate(grid_points):
            if i % 50 == 0:  # Progress indicator
                print(f"Processing point {i+1}/{len(grid_points)}...")
            
            # Predict using real-world data
            metrics = self.predict_metrics_at_point(
                point[0], point[1], start_lat, start_lon, dest_lat, dest_lon,
                spreading_factor, frequency_mhz, tx_power_dbm
            )
            grid_metrics.append(metrics)
        
        # Find nearest grid points to start and destination
        start_idx = np.argmin([distance.euclidean((start_lat, start_lon), point) 
                              for point in grid_points])
        dest_idx = np.argmin([distance.euclidean((dest_lat, dest_lon), point) 
                             for point in grid_points])
        
        print(f"Start point: {grid_points[start_idx]}")
        print(f"Destination point: {grid_points[dest_idx]}")
        
        # Find optimal path using A*
        print("Running A* search...")
        path_indices = self.a_star_search(start_idx, dest_idx, grid_points, grid_metrics)
        
        if path_indices is None:
            print("No path found using A*! Using direct path.")
            path_indices = [start_idx, dest_idx]
        
        # Extract path coordinates and metrics
        path_coords = [grid_points[i] for i in path_indices]
        path_metrics = [grid_metrics[i] for i in path_indices]
        
        path_df = pd.DataFrame(path_metrics)
        path_df['latitude'] = [coord[0] for coord in path_coords]
        path_df['longitude'] = [coord[1] for coord in path_coords]
        
        return path_df, grid_points, grid_metrics
    
    def create_direct_path(self, start_lat, start_lon, dest_lat, dest_lon, num_points=20,
                          spreading_factor=7, frequency_mhz=868, tx_power_dbm=14):
        """Create a direct (straight-line) path for comparison using real-world data"""
        lats = np.linspace(start_lat, dest_lat, num_points)
        lons = np.linspace(start_lon, dest_lon, num_points)
        
        direct_metrics = []
        
        for lat, lon in zip(lats, lons):
            # Predict using real-world data
            metrics = self.predict_metrics_at_point(
                lat, lon, start_lat, start_lon, dest_lat, dest_lon,
                spreading_factor, frequency_mhz, tx_power_dbm
            )
            metrics['latitude'] = lat
            metrics['longitude'] = lon
            direct_metrics.append(metrics)
        
        return pd.DataFrame(direct_metrics)

### Visualization Functions

In [ ]:
def visualize_all_paths_comparison(all_optimized_paths, all_direct_paths, grid_points_list, 
                                 grid_metrics_list, start_coords, end_coords, spreading_factor,
                                 frequency_mhz, tx_power_dbm):
    """Create comprehensive visualization comparing all models"""
    
    # Calculate center
    all_lats = [start_coords[0], end_coords[0]]
    all_lons = [start_coords[1], end_coords[1]]
    
    for path_df in all_optimized_paths:
        all_lats.extend(path_df['latitude'])
        all_lons.extend(path_df['longitude'])
    
    center_lat = (min(all_lats) + max(all_lats)) / 2
    center_lon = (min(all_lons) + max(all_lons)) / 2
    
    # Create map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=14)
    
    # Colors for different models
    colors = ['blue', 'green', 'red']
    model_names = ['Random Forest', 'XGBoost', 'Neural Network (PyTorch)']
    
    # Add grid heatmap (using the first model's metrics)
    if grid_metrics_list:
        grid_metrics = []
        for i, point in enumerate(grid_points_list[0]):
            if i < len(grid_metrics_list[0]):
                pdr = grid_metrics_list[0][i]['PDR']
                grid_metrics.append([point[0], point[1], pdr])
        
        HeatMap(grid_metrics, min_opacity=0.2, radius=8, blur=6, max_zoom=15).add_to(m)
    
    # Add paths for all models
    for i, (opt_path, direct_path) in enumerate(zip(all_optimized_paths, all_direct_paths)):
        color = colors[i]
        model_name = model_names[i]
        
        # Optimized path
        opt_coords = [[row['latitude'], row['longitude']] for _, row in opt_path.iterrows()]
        folium.PolyLine(
            locations=opt_coords,
            color=color,
            weight=4,
            opacity=0.8,
            popup=f'{model_name} Optimized Path'
        ).add_to(m)
        
        # Direct path (dashed)
        direct_coords = [[row['latitude'], row['longitude']] for _, row in direct_path.iterrows()]
        folium.PolyLine(
            locations=direct_coords,
            color=color,
            weight=2,
            opacity=0.5,
            dash_array='10,10',
            popup=f'{model_name} Direct Path'
        ).add_to(m)
    
    # Add start and end markers
    folium.Marker(
        location=start_coords,
        popup='Start',
        icon=folium.Icon(color='black', icon='play')
    ).add_to(m)
    
    folium.Marker(
        location=end_coords,
        popup='End',
        icon=folium.Icon(color='black', icon='stop')
    ).add_to(m)
    
    # Add legend
    legend_html = f'''
    <div style="position: fixed; 
                top: 10px; right: 10px; width: 220px; height: 160px; 
                background-color: white; border:2px solid grey; z-index:9999; 
                font-size:12px; padding: 10px">
    <p><strong>LoRa Parameters</strong></p>
    <p>SF: {spreading_factor}</p>
    <p>Freq: {frequency_mhz} MHz</p>
    <p>Tx Power: {tx_power_dbm} dBm</p>
    <p><strong>Paths</strong></p>
    <p><i class="fa fa-minus" style="color:blue"></i> Random Forest</p>
    <p><i class="fa fa-minus" style="color:green"></i> XGBoost</p>
    <p><i class="fa fa-minus" style="color:red"></i> Neural Network</p>
    <p>Solid: Optimized | Dashed: Direct</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    return m

def plot_comprehensive_comparison(all_optimized_paths, all_direct_paths, spreading_factor,
                                frequency_mhz, tx_power_dbm):
    """Plot comprehensive comparison of all models"""
    
    model_names = ['Random Forest', 'XGBoost', 'Neural Network (PyTorch)']
    colors = ['blue', 'green', 'red']
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'Comprehensive Path Comparison - SF={spreading_factor}, Freq={frequency_mhz}MHz, TxPower={tx_power_dbm}dBm', fontsize=16)
    
    # Calculate distances for each path
    def calculate_distances(df):
        distances = [0]
        for i in range(1, len(df)):
            lat1, lon1 = df.iloc[i-1]['latitude'], df.iloc[i-1]['longitude']
            lat2, lon2 = df.iloc[i]['latitude'], df.iloc[i]['longitude']
            dist = distance.euclidean((lat1, lon1), (lat2, lon2))
            distances.append(distances[-1] + dist)
        return distances
    
    # Plot PDR comparison
    ax = axes[0, 0]
    for i, (opt_path, direct_path) in enumerate(zip(all_optimized_paths, all_direct_paths)):
        opt_distances = calculate_distances(opt_path)
        direct_distances = calculate_distances(direct_path)
        
        opt_path['distance'] = opt_distances
        direct_path['distance'] = direct_distances
        
        ax.plot(opt_path['distance'], opt_path['PDR'], 
               color=colors[i], linewidth=2, label=f'{model_names[i]} Optimized')
        ax.plot(direct_path['distance'], direct_path['PDR'], 
               color=colors[i], linewidth=1, linestyle='--', alpha=0.7, 
               label=f'{model_names[i]} Direct')
    
    ax.set_xlabel('Distance along path')
    ax.set_ylabel('PDR')
    ax.set_title('Packet Delivery Rate Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot RSSI comparison
    ax = axes[0, 1]
    for i, (opt_path, direct_path) in enumerate(zip(all_optimized_paths, all_direct_paths)):
        ax.plot(opt_path['distance'], opt_path['RSSI'], 
               color=colors[i], linewidth=2, label=f'{model_names[i]} Optimized')
        ax.plot(direct_path['distance'], direct_path['RSSI'], 
               color=colors[i], linewidth=1, linestyle='--', alpha=0.7)
    
    ax.set_xlabel('Distance along path')
    ax.set_ylabel('RSSI (dBm)')
    ax.set_title('Signal Strength Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot SNR comparison
    ax = axes[1, 0]
    for i, (opt_path, direct_path) in enumerate(zip(all_optimized_paths, all_direct_paths)):
        ax.plot(opt_path['distance'], opt_path['SNR'], 
               color=colors[i], linewidth=2, label=f'{model_names[i]} Optimized')
        ax.plot(direct_path['distance'], direct_path['SNR'], 
               color=colors[i], linewidth=1, linestyle='--', alpha=0.7)
    
    ax.set_xlabel('Distance along path')
    ax.set_ylabel('SNR (dB)')
    ax.set_title('Signal-to-Noise Ratio Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot average metrics comparison
    ax = axes[1, 1]
    metrics = ['PDR', 'RSSI', 'SNR']
    x = np.arange(len(metrics))
    width = 0.15
    
    for i, model_name in enumerate(model_names):
        opt_path = all_optimized_paths[i]
        direct_path = all_direct_paths[i]
        
        opt_avgs = [opt_path[metric].mean() for metric in metrics]
        direct_avgs = [direct_path[metric].mean() for metric in metrics]
        
        ax.bar(x - width + i*width*2, opt_avgs, width, 
               color=colors[i], alpha=0.8, label=f'{model_name} Opt')
        ax.bar(x + width + i*width*2, direct_avgs, width, 
               color=colors[i], alpha=0.4, label=f'{model_name} Dir')
    
    ax.set_xlabel('Metrics')
    ax.set_ylabel('Average Value')
    ax.set_title('Average Metrics Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'comprehensive_path_comparison_sf{spreading_factor}.png', dpi=300, bbox_inches='tight')
    plt.show()

### Main Function

In [ ]:
def run_comprehensive_optimization(start_lat, start_lon, dest_lat, dest_lon,
                                spreading_factor=7, frequency_mhz=868, tx_power_dbm=14):
    """Run optimization with all models using real-world data"""
    
    print("\n=== COMPREHENSIVE PATH OPTIMIZATION WITH REAL-WORLD DATA ===")
    print(f"Start: ({start_lat}, {start_lon})")
    print(f"Destination: ({dest_lat}, {dest_lon})")
    print(f"Parameters: SF={spreading_factor}, Freq={frequency_mhz}MHz, TxPower={tx_power_dbm}dBm")
    
    # Initialize optimizers for all models
    optimizers = {}
    for model_name, model in models.items():
        optimizers[model_name] = AStarPathOptimizer(
            model, scaler, y_scaler, model_name
        )
    
    # Find optimal paths using all models
    all_optimized_paths = []
    all_direct_paths = []
    grid_points_list = []
    grid_metrics_list = []
    
    for model_name, optimizer in optimizers.items():
        print(f"\n--- {model_name} ---")
        
        # Find optimized path
        opt_path, grid_points, grid_metrics = optimizer.find_optimal_path_astar(
            start_lat, start_lon, dest_lat, dest_lon,
            spreading_factor, frequency_mhz, tx_power_dbm
        )
        
        # Create direct path for comparison
        direct_path = optimizer.create_direct_path(
            start_lat, start_lon, dest_lat, dest_lon, 20,
            spreading_factor, frequency_mhz, tx_power_dbm
        )
        
        # Calculate improvement
        opt_avg_pdr = opt_path['PDR'].mean()
        direct_avg_pdr = direct_path['PDR'].mean()
        improvement = ((opt_avg_pdr - direct_avg_pdr) / direct_avg_pdr * 100) if direct_avg_pdr > 0 else 0
        
        print(f"Average PDR - Optimized: {opt_avg_pdr:.3f}, Direct: {direct_avg_pdr:.3f}")
        print(f"Improvement: {improvement:.1f}%")
        
        all_optimized_paths.append(opt_path)
        all_direct_paths.append(direct_path)
        grid_points_list.append(grid_points)
        grid_metrics_list.append(grid_metrics)
    
    # Create comprehensive visualization
    print("\nCreating comprehensive visualization...")
    comparison_map = visualize_all_paths_comparison(
        all_optimized_paths, all_direct_paths, grid_points_list, 
        grid_metrics_list, (start_lat, start_lon), (dest_lat, dest_lon),
        spreading_factor, frequency_mhz, tx_power_dbm
    )
    comparison_map.save(f'comprehensive_path_comparison_sf{spreading_factor}.html')
    
    # Plot comprehensive comparison
    plot_comprehensive_comparison(
        all_optimized_paths, all_direct_paths, spreading_factor, frequency_mhz, tx_power_dbm
    )
    
    # Create summary table
    summary_data = []
    model_names = ['Random Forest', 'XGBoost', 'Neural Network (PyTorch)']
    
    for i, model_name in enumerate(model_names):
        opt_path = all_optimized_paths[i]
        direct_path = all_direct_paths[i]
        
        summary_data.append({
            'Model': model_name,
            'Optimized_PDR': opt_path['PDR'].mean(),
            'Direct_PDR': direct_path['PDR'].mean(),
            'PDR_Improvement_%': ((opt_path['PDR'].mean() - direct_path['PDR'].mean()) / 
                                 direct_path['PDR'].mean() * 100) if direct_path['PDR'].mean() > 0 else 0,
            'Optimized_RSSI': opt_path['RSSI'].mean(),
            'Direct_RSSI': direct_path['RSSI'].mean(),
            'Optimized_SNR': opt_path['SNR'].mean(),
            'Direct_SNR': direct_path['SNR'].mean()
        })
    
    summary_df = pd.DataFrame(summary_data)
    print("\n=== SUMMARY TABLE ===")
    print(summary_df.to_string(index=False))
    
    # Save summary
    summary_df.to_csv(f'optimization_summary_sf{spreading_factor}.csv', index=False)
    
    return all_optimized_paths, all_direct_paths, summary_df

# Function to save PyTorch model
def save_pytorch_model():
    """Save the trained PyTorch model"""
    torch.save({
        'model_state_dict': models['Neural Network'].state_dict(),
        'scaler': scaler,
        'y_scaler': y_scaler,
        'input_size': X_train_scaled.shape[1],
        'output_size': y_train_scaled.shape[1],
        'features': features  # Save the physical features used
    }, 'pytorch_lora_model.pth')
    print("\nPyTorch model saved as 'pytorch_lora_model.pth'")

# Save the PyTorch model
save_pytorch_model()

if __name__ == "__main__":
    # Example from dataset 1
    print("=== Example 1: Using coordinates from dataset 1 ===")
    start_lat, start_lon = 28.98947, 50.836233
    dest_lat, dest_lon = 28.992581, 50.835009
    
    all_opt_paths, all_dir_paths, summary_df = run_comprehensive_optimization(
        start_lat, start_lon, dest_lat, dest_lon
    )
    
    # Example from dataset 2
    print("\n=== Example 2: Using coordinates from dataset 2 ===")
    start_lat, start_lon = 42.47050775755835, -9.001788139343253
    dest_lat, dest_lon = 42.47, -9.01
    
    all_opt_paths, all_dir_paths, summary_df = run_comprehensive_optimization(
        start_lat, start_lon, dest_lat, dest_lon
    )

### Testing the Model

In [ ]:
def test_custom_path():
    print("\n=== TEST CUSTOM PATH WITH ALL MODELS ===")
    try:
        print("Enter start coordinates:")
        start_lat = float(input("Start latitude: "))
        start_lon = float(input("Start longitude: "))
        
        print("\nEnter destination coordinates:")
        dest_lat = float(input("Destination latitude: "))
        dest_lon = float(input("Destination longitude: "))
        
        print("\nEnter LoRa parameters (press Enter for defaults):")
        spreading_factor = input("Spreading Factor (7-12, default=7): ")
        spreading_factor = int(spreading_factor) if spreading_factor else 7
        
        frequency_mhz = input("Frequency MHz (default=868): ")
        frequency_mhz = float(frequency_mhz) if frequency_mhz else 868
        
        tx_power_dbm = input("TX Power dBm (default=14): ")
        tx_power_dbm = float(tx_power_dbm) if tx_power_dbm else 14
        
        # Run optimization
        all_opt_paths, all_dir_paths, summary_df = run_comprehensive_optimization(
            start_lat, start_lon, dest_lat, dest_lon,
            spreading_factor, frequency_mhz, tx_power_dbm
        )
        
        print("\nCustom path optimization complete!")
        
        # Show best performing model
        best_model_idx = summary_df['PDR_Improvement_%'].idxmax()
        best_model = summary_df.loc[best_model_idx, 'Model']
        best_improvement = summary_df.loc[best_model_idx, 'PDR_Improvement_%']
        
        print(f"\nBest performing model: {best_model} with {best_improvement:.1f}% PDR improvement")
        
        # Show optimized path details for best model
        best_opt_path = all_opt_paths[best_model_idx]
        print(f"\nOptimized path details for {best_model}:")
        print(best_opt_path[['latitude', 'longitude', 'PDR', 'RSSI', 'SNR', 'path_loss']].head())
        
    except ValueError:
        print("Invalid input. Please enter valid numerical coordinates.")
    
# Uncomment to test custom path
# test_custom_path()